# Votations fédérales suisses — Exploration des données brutes (OFS px-x-1703030000_101)

**Objectif :** explorer la couverture et la qualité du tableau historique de l'OFS
(votations populaires par commune depuis 1960), tel que collecté, avant tout traitement.

- Collecte : `python -m scripts.ofs_votes.collect_votations_federales` → `data/raw/votations_communes_1960_2026_brut.csv`
- Documentation de la source : `docs/datasets/votations_px_101/README.md`
- Ce notebook **ne lit que `data/raw/`** et ne modifie rien. Le nettoyage et la mise en forme
  sont faits ailleurs, avec sortie dans `data/processed/`.

## 1. Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## 2. Chargement des données brutes

In [2]:
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Racine du dépôt introuvable (aucun dossier .git en remontant depuis le dossier courant).")

RAW_CSV = PROJECT_ROOT / "data" / "raw" / "votations_communes_1960_2026_brut.csv"
if not RAW_CSV.exists():
    raise FileNotFoundError(f"{RAW_CSV} introuvable : lancez d'abord python -m scripts.ofs_votes.collect_votations_federales")

df_raw = pd.read_csv(RAW_CSV)
df_raw.shape

(8452451, 4)

## 3. Vue d'ensemble des données brutes

Format **long** (une ligne par commune/canton/district × objet de votation × indicateur),
tel que renvoyé par l'API. Les colonnes sont identifiées par leur position, une seule fois,
pour être réutilisées dans tout le notebook.

In [3]:
col_geo, col_objet, col_resultat = df_raw.columns[:3]
col_valeur = "value"

print("Colonne géographique :", col_geo)
print("Colonne date/objet   :", col_objet)
print("Colonne indicateur   :", col_resultat)
print("Colonne valeur       :", col_valeur)

df_raw.head()

Colonne géographique : Canton (-) / District (>>) / Commune (......)
Colonne date/objet   : Date et objet
Colonne indicateur   : Résultat
Colonne valeur       : value


,Canton (-) / District (>>) / Commune (......),Date et objet,Résultat,value
0,Suisse,2026-06-14 Modification de la loi fédérale sur...,Electeurs inscrits,5669113.00
1,Suisse,2026-06-14 Modification de la loi fédérale sur...,Bulletins rentrés,3303080.00
2,Suisse,2026-06-14 Modification de la loi fédérale sur...,Participation en %,58.26
3,Suisse,2026-06-14 Modification de la loi fédérale sur...,Bulletins valables,3222782.00
4,Suisse,2026-06-14 Modification de la loi fédérale sur...,Oui,1690622.00


### Structure hiérarchique des données

Pour **chaque objet voté**, le tableau contient les résultats à 4 niveaux géographiques emboîtés :
```
Suisse
└─ Canton
└─ District
└─ Commune
```

Ce niveau est encodé dans le **texte** de la colonne géographique par un préfixe :

| Préfixe | Niveau |
|---|---|
| *(aucun)* | Suisse |
| `-` | Canton |
| `>>` | District |
| `......` | Commune |

Pour une unité géographique et un objet donnés, il y a **7 lignes** (une par indicateur :
Electeurs inscrits, Bulletins rentrés, Participation en %, Bulletins valables, Oui, Non, Oui en %).

Deux particularités à anticiper :
- certains petits cantons (Uri, Zoug, Glaris...) n'ont pas de vrai niveau district : leurs communes sont rattachées directement au canton ;
- chaque canton a aussi des entités techniques au même niveau que les districts (Suisses de l'étranger, votes non attribués...), qui ne sont pas de vraies communes.

Cette hiérarchie Suisse → canton → district → commune, présente dans un seul tableau, permettra
l'analyse « à contre-courant » du cahier des charges (comparer une commune au reste de son canton).

## 4. Exploration et contrôle qualité

### 4.1 Couverture temporelle et géographique

In [4]:
n_objets = df_raw[col_objet].nunique()
n_geo = df_raw[col_geo].nunique()
dates = sorted(df_raw[col_objet].str.extract(r"(\d{4}-\d{2}-\d{2})")[0].dropna().unique())

print(f"{n_objets} objets de votation distincts")
print(f"{n_geo} unités géographiques distinctes (Suisse, cantons, districts, communes)")
print(f"{len(dates)} dates de votation, de {dates[0]} à {dates[-1]}")

511 objets de votation distincts
2363 unités géographiques distinctes (Suisse, cantons, districts, communes)
191 dates de votation, de 1960-05-29 à 2026-06-14


### 4.2 Valeurs manquantes

Deux causes distinctes, documentées dans les métadonnées du tableau OFS et confirmées ci-dessous :

1. **1960-1980** : couverture incomplète pour les cantons FR, AG et TI (annoncé par l'OFS) — gros volume de manquants concentré sur cette période.
2. **Après 1981** : manquants résiduels constants (même nombre d'une année à l'autre), qui ne sont **pas de vraies données absentes** mais des catégories techniques du référentiel — Suisses de l'étranger (`*-Ausland-CH`), votes non attribués (`*-Anderes`), vote par correspondance compté à part (`*-Korrespondenzweg`), communes sans bureau de vote propre (« Urne commune »).

In [5]:
manquants = df_raw[df_raw[col_valeur].isna()]
par_annee = manquants[col_objet].str.extract(r"(\d{4})")[0].value_counts().sort_index()

print(f"{len(manquants):,} valeurs manquantes ({len(manquants) / len(df_raw):.1%} des lignes)")
print("\nValeurs manquantes par année (extrait) :")
print(par_annee.head(10))
print("...")
print(par_annee.tail(10))

689,153 valeurs manquantes (8.2% des lignes)

Valeurs manquantes par année (extrait) :
0
1960     7966
1961    15691
1962    15910
1963    11920
1964    12220
1965    12231
1966     7898
1967     3993
1968     7926
1969     7962
Name: count, dtype: int64
...
0
2017    3185
2018    4480
2019    1344
2020    4032
2021    5733
2022    4851
2023    1323
2024    5292
2025    2205
2026    3528
Name: count, dtype: int64


In [6]:
post_1981 = manquants[manquants[col_objet].str.extract(r"(\d{4})")[0].astype(int) >= 1981]
post_1981[col_geo].value_counts().head(15)

Canton (-) / District (>>) / Commune (......)
>> ZH-Anderes                                                                                 2751
......ZH-Anderes                                                                              2751
......Meienried (Urne commune, emplacement: Büren an der Aare          )                      2751
......Hellsau (Urne commune, emplacement: Höchstetten                )                        2751
......Deisswil bei Münchenbuchsee (Urne commune, emplacement: Wiggiswil                  )    2751
......Oberhünigen (Urne commune, emplacement: Zäziwil                    )                    2751
......Niedermuhlern (Urne commune, emplacement: Wald (BE)                  )                  2751
>> BE-Ausland-CH                                                                              2751
......BE-Ausland-CH                                                                           2751
>> BE-Anderes                                                  

### 4.3 Doublons

In [7]:
doublons = df_raw.duplicated(subset=[col_geo, col_objet, col_resultat], keep=False)
print(f"{doublons.sum()} lignes dupliquées sur {len(df_raw):,} (clé : géo × objet × indicateur)")

0 lignes dupliquées sur 8,452,451 (clé : géo × objet × indicateur)
